# Step 19.4 — Apple Silicon Resource Control

本节只覆盖 unified memory awareness、avoid Swap、conservative batch/model sizing、sustained heat vs short bursts。承接18.5内存计数、18.2求值边界与19.3稳定性；不运行训练或资源测量。

19.3实际提交 `d193f85640bf9f36e79fcaf035612f76ba133895`及完整初始/修正回复已独立核心静态验收PASS，累计修正1/2。这是材料验收，不是学员掌握或运行反馈。

本Notebook代码单元均未执行且无输出；19.4初始生成0/2。完成即停在19.4，不进入19.5。

## 0. 自包含与范围

本母版自带导入、19.1候选配置、可选数据获取定义、本地读取与split、sampler、完整MiniGPT、loss、稳定性helper和资源控制配置函数。默认不下载语料、不构造模型、不执行训练、不读取系统资源。未来练习使用Python 3.12、项目`.venv`与固定`mlx==0.32.2`。

资源控制是风险管理，不是对零Swap、低温、固定速度或必然收敛的保证。

In [ ]:
import math
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_map

baseline_candidate = {'model_width':64,'layers':2,'heads':4,'block_size':64,'batch_size':4,'learning_rate':3e-4}
seeds = {'model':2028,'train':2026,'validation':2027}
def make_model_kwargs(c,vocab_size):
    return {'vocab_size':vocab_size,'dims':c['model_width'],'num_heads':c['heads'],'num_blocks':c['layers'],'max_seq_len':c['block_size']}
def make_optimizer_kwargs(c):
    return {'learning_rate':c['learning_rate'],'betas':[0.9,0.999],'eps':1e-8,'bias_correction':True} # explicit course choice; MLX default False

## 1. Unified memory：哪些资源会共享？

Apple Silicon采用unified memory architecture：CPU和GPU可以访问共享的系统内存池。模型参数、激活、中间数组、MLX分配器缓存、Python对象、编译器/驱动资源与其他应用都可能影响整机压力，但它们不属于同一个计数器。

18.5的`get_active_memory()`、`get_cache_memory()`、`get_peak_memory()`是MLX分配器的计数；它们不等于Activity Monitor的Memory Used、Memory Pressure或Swap Used。Cache memory是MLX保留的闲置buffer，不是macOS Swap。

## 2. 逻辑估算不等于真实峰值

若有P个float32参数，单份参数的逻辑payload为4P bytes。Adam还保存m和v，另有梯度、激活、attention scores、中间结果、编译和缓存等开销。逻辑payload是预算起点，不是进程峰值或整机内存测量。

本候选C=64、H=4、T=64、B=4；attention scores的逻辑shape为(B,H,T,T)，元素数为BHT²。它随B和T变化，但不能据此断言实际内存按同一比例增长，因为生命周期、融合、view、分配器复用与其他资源都会影响结果。

In [ ]:
def logical_float32_bytes(shape):
    """Arithmetic estimate only; does not allocate or measure memory."""
    elements = math.prod(shape)
    return elements * 4

# Paper example: (4, 4, 64, 64) contains 65,536 float32 elements.
# Its logical payload is 65,536 * 4 bytes = 262,144 bytes = 0.25 MiB.
# This is not a measured buffer size or peak-memory prediction.

## 3. Conservative sizing：先减少并发工作量

Batch size B决定每步同时处理多少个窗口；block size T决定每行的上下文长度。对于显式attention scores，逻辑元素数为BHT²，因此降低B线性减少该张量的元素数，降低T则以平方关系减少它的元素数（其余维度不变）。这只是形状推导，不是速度/峰值保证。

Model width C与层数N还影响参数和中间激活。当前候选不是已测得的安全上限；若未来资源紧张，应一次只改一个配置维度，并记录改动，不要同时改变所有参数后把差异归因于某一个因素。

In [ ]:
def propose_resource_candidate(config, *, batch_size=None, block_size=None, model_width=None, layers=None):
    """Return a new candidate dictionary; does not instantiate a model or change system settings."""
    candidate = dict(config)
    if batch_size is not None: candidate['batch_size'] = batch_size
    if block_size is not None:
        candidate['block_size'] = block_size
        candidate['max_seq_len'] = block_size
    if model_width is not None: candidate['model_width'] = model_width
    if layers is not None: candidate['layers'] = layers
    if candidate['model_width'] % candidate['heads'] != 0:
        raise ValueError('model_width must be divisible by heads')
    if min(candidate['batch_size'], candidate['block_size'], candidate['model_width'], candidate['layers']) < 1:
        raise ValueError('size parameters must be positive')
    return candidate

# Example call is intentionally not made in this master.

## 4. Avoid Swap：监测压力，不做虚假保证

Swap是macOS在内存压力下使用的磁盘交换空间。MLX的active/cache/peak计数器不提供Swap Used，也不能证明整机不会换页。即使MLX cache被清理，仍存活的模型参数、梯度、激活、Python对象和其他应用也会占用资源；操作系统还可能保留可复用缓存。

未来人工观察应同时区分：MLX allocator counters、Activity Monitor的Memory Pressure/Memory Used/Swap Used，以及当时其他应用的负载。本节不读取这些指标、不更改内存限制或系统权限。

In [ ]:
def mlX_memory_snapshot_bytes():
    """MLX allocator counters only; all values are bytes, not system RAM or Swap."""
    return {
        'active_bytes': mx.get_active_memory(),
        'cache_bytes': mx.get_cache_memory(),
        'peak_bytes': mx.get_peak_memory(),
    }

# Definition only. Do not call during material preparation.

## 5. 求值边界与状态生命周期

惰性数组可能尚未完成数值工作。要把某个训练步作为观察边界，先对该步的loss、当前模型参数和optimizer state执行`mx.eval(...)`，再读取MLX计数器。只求值loss不能保证新参数和Adam状态也完成。

求值不会释放仍被引用的数组，也不保证操作系统占用立即下降。清理MLX allocator cache（如`mx.clear_cache()`）只针对闲置的MLX buffer，不会清空编译图缓存、删除模型状态或保证Swap为零；本节不调用它。

## 6. Sustained heat vs short bursts

**Short burst（短时突发）**是有限时间的工作负载；**sustained workload（持续负载）**会让CPU/GPU长时间保持工作。Apple Silicon温度与功耗还受环境温度、散热设计、充电状态、其他进程、芯片型号和系统策略影响。

一次短调用正常结束，不证明长时间训练不会降频或积累热量；同样，模型小也不保证整机低温。本节没有温度传感器读数、功耗测量、风扇状态或长时训练证据。

## 7. 现有模型和数据依赖恢复（仅定义）

后续若要实际练习，先定位真实仓库和本地缓存；可选获取默认关闭。缺失语料时明确停止。以下定义不读取或下载数据。

In [ ]:
SOURCE_URL='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
def find_project_root(start):
    current=Path(start).resolve()
    for p in [current]+list(current.parents):
        if (p/'pyproject.toml').is_file() and (p/'notebooks'/'82_step19_1_hyperparameter_baseline.ipynb').is_file(): return p
    raise FileNotFoundError('Set PROJECT_ROOT to the repository root.')
def acquire_tiny_shakespeare(path,source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True,exist_ok=True)
    with urlopen(source_url,timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text,chars,stoi,itos,mx.array([stoi[ch] for ch in text],dtype=mx.int32)
def split_train_validation(stream,fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Invalid stream/split')
    i=int(stream.shape[0]*fraction)
    if i<=0 or i>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:i],stream[i:],i

## 8. Sampler、完整MiniGPT与loss定义

工作副本按顺序恢复：本地bootstrap→split→sampler→模型→loss。X/Y shape为(B,T)，logits为(B,T,V)，flatten为(B*T,V)/(B*T,)，交叉熵后mean返回标量。Y已shift。完整模型类沿用通过的15.4定义，不以toy模型替代。

In [ ]:
def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


# Definition only. No batch is sampled in this cell.


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


In [ ]:
def loss_fn(model,X,Y):
    logits=model(X); B,T,V=logits.shape
    flat_logits=mx.reshape(logits,(B*T,V)); flat_targets=mx.reshape(Y,(B*T,))
    losses=nn.losses.cross_entropy(flat_logits,flat_targets,axis=-1,reduction='none')
    return mx.mean(losses)

## 9. Adam与训练状态（定义，不执行）

Adam状态包括每个参数对应的m/v与step。梯度范数、优化器更新和系统资源是不同概念。本节只定义匹配setup，不实例化模型、不初始化训练状态、不调用optimizer。

In [ ]:
def make_optimizer_kwargs(config):
    return {'learning_rate':config['learning_rate'],'betas':[0.9,0.999],'eps':1e-8,'bias_correction':True} # explicit course choice; MLX default False

# Future setup, intentionally not executed:
# model = MiniGPT(**make_model_kwargs(baseline_candidate, vocab_size))
# loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
# optimizer = optim.Adam(**make_optimizer_kwargs(baseline_candidate))
# optimizer.init(model.trainable_parameters())
# training_state = [model.state, optimizer.state]

## 10. 本节边界与后续

本节只解释统一内存、避免Swap的风险意识、保守尺寸选择、求值与状态生命周期、短时突发与持续热负载的区别。所有数值均为公式推导或配置候选，没有系统读数、温度、训练或性能结果。

19.3实际提交`d193f85640bf9f36e79fcaf035612f76ba133895`已独立核心静态验收PASS，累计修正1/2；19.2累计1/2。19.4初始生成0/2。本轮不运行Notebook、训练、代码测试、实验、采样、测量，不读取/下载语料，不改变系统资源或权限。停在19.4，不进入19.5；后续按请求逐小节至大Step20全部材料完成后停止，不进入21。